# Token 与 Tokenizer：从字符串到模型输入

这个 notebook 用纯 Python 从零解释 token、字符、词、token id、词表、padding 和 truncation。代码是教学实现，不复刻某个商业模型的 tokenizer，但每个中间结果都可以直接观察。

## 学习目标

1. 区分 character、Unicode code point、UTF-8 byte、word 与 token。
2. 理解 `文本 -> 规范化 -> 切分 -> token id -> embedding` 的数据流。
3. 观察 token 粒度如何影响序列长度、attention 计算和 KV cache。
4. 正确实现 special token、padding、truncation 与 attention mask。
5. 知道如何评估 tokenizer，而不是只看一个示例的 token 数。

## 1. character、word 与 token

- **character** 常被口语化为字符，但计算机中还要区分 Unicode code point、UTF-8 byte 和用户看到的 grapheme cluster。
- **word** 是语言学单位。英文常有空格边界，中文没有天然空格，代码标识符又有自己的结构。
- **token** 是某个具体 tokenizer 根据词表与规则生成的离散单位。一个 token 可以是词、子词、汉字、字节片段、空格或标点。

三者没有固定的一一对应关系。模型真正处理的是 token id 序列；id 再从 embedding 矩阵中查出向量。

In [ ]:
samples = ["Token", "中文", "é", "e\u0301", "🙂"]  # 计算并保存当前步骤的中间状态。

for text in samples:  # 遍历输入元素以累积或检查结果。
    print({  # 执行当前语句以推进本节示例。
        "文本": text,  # 执行当前语句以推进本节示例。
        "Python_len": len(text),          # Unicode code point 数
        "code_points": [f"U+{ord(ch):04X}" for ch in text],  # 执行当前语句以推进本节示例。
        "UTF8字节数": len(text.encode("utf-8")),  # 执行当前语句以推进本节示例。
        "UTF8字节": list(text.encode("utf-8")),  # 执行当前语句以推进本节示例。
    })  # 执行当前语句以推进本节示例。


## 2. Tokenizer 的完整流水线

典型流程可以拆成：

```text
原始字符串
  -> normalization（Unicode、大小写、空白等，可选）
  -> pre-tokenization（空格、标点或字节边界）
  -> BPE / WordPiece / Unigram 等子词模型
  -> 加入 BOS/EOS 等 special token
  -> 词表查找得到 token id
  -> padding / truncation 与 attention mask
  -> embedding lookup
```

只要 normalization 出现多对一变换，例如小写化或合并连续空格，`decode(encode(text)) == text` 就不再严格成立。下面先做一个保留标点的极简 tokenizer。

In [ ]:
import re  # 导入本单元所需的依赖。
from collections import Counter  # 导入本单元所需的依赖。

def simple_tokenize(text: str) -> list[str]:  # 定义本节可复用的核心函数。
    # 教学规则：连续英文/数字、单个汉字、其他非空白符号分别作为 token。
    return re.findall(r"[A-Za-z]+|[0-9]+|[\u4e00-\u9fff]|[^\w\s]", text)  # 返回当前分支计算出的结果。

corpus = [  # 计算并保存当前步骤的中间状态。
    "Tokenization helps models process text.",  # 执行当前语句以推进本节示例。
    "模型处理 token，也处理标点！",  # 执行当前语句以推进本节示例。
    "Token token TOKEN",  # 执行当前语句以推进本节示例。
]  # 执行当前语句以推进本节示例。

all_tokens = [tok for text in corpus for tok in simple_tokenize(text)]  # 计算并保存当前步骤的中间状态。
counts = Counter(all_tokens)  # 计算并保存当前步骤的中间状态。
vocab_tokens = ["<pad>", "<unk>", "<bos>", "<eos>"] + sorted(counts)  # 计算并保存当前步骤的中间状态。
token_to_id = {token: idx for idx, token in enumerate(vocab_tokens)}  # 计算并保存当前步骤的中间状态。
id_to_token = {idx: token for token, idx in token_to_id.items()}  # 计算并保存当前步骤的中间状态。

def encode(text: str, add_special_tokens: bool = True) -> list[int]:  # 定义本节可复用的核心函数。
    ids = [token_to_id.get(tok, token_to_id["<unk>"]) for tok in simple_tokenize(text)]  # 计算并保存当前步骤的中间状态。
    return [token_to_id["<bos>"], *ids, token_to_id["<eos>"]] if add_special_tokens else ids  # 返回当前分支计算出的结果。

def decode(ids: list[int], skip_special_tokens: bool = True) -> list[str]:  # 定义本节可复用的核心函数。
    special = {"<pad>", "<bos>", "<eos>"}  # 计算并保存当前步骤的中间状态。
    tokens = [id_to_token.get(idx, "<unk>") for idx in ids]  # 计算并保存当前步骤的中间状态。
    return [tok for tok in tokens if not (skip_special_tokens and tok in special)]  # 返回当前分支计算出的结果。

text = "模型处理 Token！"  # 计算并保存当前步骤的中间状态。
ids = encode(text)  # 计算并保存当前步骤的中间状态。
print("tokens:", simple_tokenize(text))  # 执行当前语句以推进本节示例。
print("ids:   ", ids)  # 执行当前语句以推进本节示例。
print("decode 后的 token:", decode(ids))  # 执行当前语句以推进本节示例。
print("词表大小:", len(token_to_id))  # 执行当前语句以推进本节示例。


## 3. 为什么大模型常用 subword

纯词级词表无法穷举姓名、新术语、URL 和代码变量；纯字符或字节虽然覆盖稳定，却会显著拉长序列。subword 在二者之间折中：

- 高频片段合成更长 token，缩短常见文本；
- 长尾内容退化为较小单元，避免硬 OOV；
- 用有限词表获得开放组合能力。

BPE 的核心是反复合并高频相邻 pair。下面只演示一轮计数和一次不重叠合并；完整训练还需要迭代、维护词频、稳定处理并列频次并保存 merge 顺序。

In [ ]:
def pair_counts(word_freqs: dict[tuple[str, ...], int]) -> Counter:  # 定义本节可复用的核心函数。
    result = Counter()  # 计算并保存当前步骤的中间状态。
    for symbols, freq in word_freqs.items():  # 遍历输入元素以累积或检查结果。
        for left, right in zip(symbols, symbols[1:]):  # 遍历输入元素以累积或检查结果。
            result[(left, right)] += freq  # 计算并保存当前步骤的中间状态。
    return result  # 返回当前分支计算出的结果。

def merge_pair(  # 定义本节可复用的核心函数。
    word_freqs: dict[tuple[str, ...], int], pair: tuple[str, str]  # 执行当前语句以推进本节示例。
) -> dict[tuple[str, ...], int]:  # 执行当前语句以推进本节示例。
    merged_words = {}  # 计算并保存当前步骤的中间状态。
    for symbols, freq in word_freqs.items():  # 遍历输入元素以累积或检查结果。
        output, i = [], 0  # 计算并保存当前步骤的中间状态。
        while i < len(symbols):  # 在终止条件满足前持续推进状态。
            if i + 1 < len(symbols) and (symbols[i], symbols[i + 1]) == pair:  # 按当前条件选择后续控制路径。
                output.append(symbols[i] + symbols[i + 1])  # 执行当前语句以推进本节示例。
                i += 2  # 计算并保存当前步骤的中间状态。
            else:  # 处理前置条件不成立的分支。
                output.append(symbols[i])  # 执行当前语句以推进本节示例。
                i += 1  # 计算并保存当前步骤的中间状态。
        merged_words[tuple(output)] = freq  # 计算并保存当前步骤的中间状态。
    return merged_words  # 返回当前分支计算出的结果。

word_freqs = {tuple("low"): 5, tuple("lower"): 2, tuple("new"): 6}  # 计算并保存当前步骤的中间状态。
for step in range(4):  # 遍历输入元素以累积或检查结果。
    counts = pair_counts(word_freqs)  # 计算并保存当前步骤的中间状态。
    best_pair, frequency = counts.most_common(1)[0]  # 计算并保存当前步骤的中间状态。
    print(f"第 {step + 1} 轮: 合并 {best_pair}, 加权频次={frequency}")  # 计算并保存当前步骤的中间状态。
    word_freqs = merge_pair(word_freqs, best_pair)  # 计算并保存当前步骤的中间状态。
    print("  当前表示:", word_freqs)  # 执行当前语句以推进本节示例。


## 4. token 数为什么影响成本

模型按序列位置计算，而不是按原始字符计算。标准 self-attention 的分数矩阵形状为 `[batch, heads, n, n]`，其主要计算随 $n^2$ 增长；自回归推理的 KV cache 则随历史 token 数 $n$ 近似线性增长。

因此同一段文字若从 1,000 token 变成 2,000 token：

- 全局 attention 分数元素数约变为 4 倍；
- 每层 KV cache 约变为 2 倍；
- 固定上下文窗口能容纳的原始信息减少；
- 按 token 计费、prefill 延迟和截断风险都会变化。

下面只用元素个数展示量级，不分配真正的大矩阵。

In [ ]:
def attention_score_elements(batch: int, heads: int, length: int) -> int:  # 定义本节可复用的核心函数。
    return batch * heads * length * length  # 返回当前分支计算出的结果。

def kv_cache_bytes(  # 定义本节可复用的核心函数。
    layers: int, kv_heads: int, length: int, head_dim: int, bytes_per_value: int = 2  # 计算并保存当前步骤的中间状态。
) -> int:  # 执行当前语句以推进本节示例。
    # K 和 V 两份缓存；忽略对齐、分页和框架额外开销。
    return 2 * layers * kv_heads * length * head_dim * bytes_per_value  # 返回当前分支计算出的结果。

for n in [512, 1024, 2048]:  # 遍历输入元素以累积或检查结果。
    scores = attention_score_elements(batch=1, heads=32, length=n)  # 计算并保存当前步骤的中间状态。
    cache = kv_cache_bytes(layers=32, kv_heads=8, length=n, head_dim=128)  # 计算并保存当前步骤的中间状态。
    print(f"n={n:4d} | attention 元素={scores:,} | KV cache≈{cache / 2**20:.1f} MiB")  # 计算并保存当前步骤的中间状态。


## 5. Padding、Truncation 与 Mask

同一 batch 的张量需要统一长度。padding 用 `<pad>` 补齐短序列，并用 attention mask 标出真实位置；truncation 删除超过上限的 token，会永久丢失信息。

- 动态 padding 到当前 batch 最长序列，通常能减少无效计算。
- 固定 `max_length` 便于静态 shape，但 padding 浪费可能更多。
- decoder-only 批量生成常见左 padding，训练或 encoder 常见右 padding；必须遵循模型的位置 id 和实现约定。
- 截断策略要按任务选择：长文问答通常应分块或检索，而不是无脑保留开头。

In [ ]:
def pad_and_truncate(  # 定义本节可复用的核心函数。
    sequences: list[list[int]], pad_id: int, max_length: int  # 执行当前语句以推进本节示例。
) -> tuple[list[list[int]], list[list[int]]]:  # 执行当前语句以推进本节示例。
    padded, masks = [], []  # 计算并保存当前步骤的中间状态。
    for sequence in sequences:  # 遍历输入元素以累积或检查结果。
        kept = sequence[:max_length]  # 计算并保存当前步骤的中间状态。
        real_length = len(kept)  # 计算并保存当前步骤的中间状态。
        pad_length = max_length - real_length  # 计算并保存当前步骤的中间状态。
        padded.append(kept + [pad_id] * pad_length)  # 执行当前语句以推进本节示例。
        masks.append([1] * real_length + [0] * pad_length)  # 执行当前语句以推进本节示例。
    return padded, masks  # 返回当前分支计算出的结果。

sequences = [encode("模型处理 Token！"), encode("Token token TOKEN")]  # 计算并保存当前步骤的中间状态。
input_ids, attention_mask = pad_and_truncate(  # 计算并保存当前步骤的中间状态。
    sequences, pad_id=token_to_id["<pad>"], max_length=8  # 计算并保存当前步骤的中间状态。
)  # 执行当前语句以推进本节示例。
print("input_ids:")  # 执行当前语句以推进本节示例。
for row in input_ids:  # 遍历输入元素以累积或检查结果。
    print(row)  # 执行当前语句以推进本节示例。
print("attention_mask:")  # 执行当前语句以推进本节示例。
for row in attention_mask:  # 遍历输入元素以累积或检查结果。
    print(row)  # 执行当前语句以推进本节示例。


## 6. Unicode normalization 与可逆性

肉眼相同的文本可能有不同 code point 序列。NFC 会尽量合成为预组合字符，NFD 会拆成基础字符加组合标记；NFKC/NFKD 还会做兼容等价变换。只要多个原文被映射到同一规范化结果，严格原文往返就不可能。

代码、法律文本、精确引用和审计场景要特别谨慎；不要把 normalization 当成没有副作用的清洗。

In [ ]:
import unicodedata  # 导入本单元所需的依赖。

texts = ["é", "e\u0301", "ＡＢＣ", "ABC"]  # 计算并保存当前步骤的中间状态。
for text in texts:  # 遍历输入元素以累积或检查结果。
    print(  # 执行当前语句以推进本节示例。
        repr(text),  # 执行当前语句以推进本节示例。
        "NFC=", repr(unicodedata.normalize("NFC", text)),  # 计算并保存当前步骤的中间状态。
        "NFKC=", repr(unicodedata.normalize("NFKC", text)),  # 计算并保存当前步骤的中间状态。
    )  # 执行当前语句以推进本节示例。

print("前两个原始序列相等吗？", texts[0] == texts[1])  # 计算并保存当前步骤的中间状态。
print("NFC 后相等吗？", unicodedata.normalize("NFC", texts[0]) == unicodedata.normalize("NFC", texts[1]))  # 计算并保存当前步骤的中间状态。


## 7. 如何评价 tokenizer

不要只看一条英文示例。应在真实流量上按语言、代码、数字、URL、emoji 和结构化文本分桶，统计：

1. `<unk>` / OOV、字节级往返与 normalization 信息损失；
2. token/UTF-8 byte、相同语义翻译的 token 数以及 P50/P95/P99 长度；
3. 词表频率分布、低频“僵尸 token”和 embedding/LM head 参数成本；
4. encode/decode 吞吐、offset mapping、版本与 chat template 一致性；
5. 重新训练后的 loss、下游质量、训练吞吐、prefill、KV cache 和线上费用。

## 面试速答

`token 是模型视角的离散文本单位，由具体 tokenizer 决定。subword 在词表大小、序列长度和开放词汇覆盖之间折中；token 数会传导到 attention、KV cache、上下文和费用。工程上必须用目标 checkpoint 的 tokenizer 与 chat template 在真实语料上实测。`

## 练习

1. 为 `simple_tokenize` 加入可逆的空白 token，并实现字符串 decode。
2. 修改 BPE 示例，使并列最高频 pair 用稳定字典序打破平局。
3. 比较右 padding 与左 padding 的 position ids。
4. 用你自己的中英文、代码和 emoji 样本生成 token 长度分桶报告。